# Explore my portfolio — Yahoo market-data demo

A runnable tour of the APIs used in `scripts/explore_my_portfolio_2.py`, using
**real stocks and daily price history downloaded from Yahoo Finance**. Trades,
funding, fees, and account details are invented. The fixed demonstration period is
January 2024–September 2026; saved outputs show the last successful execution.

Select a Python 3.11+ kernel with ConvexPM installed and **Run All**. From the
repository root, install dependencies with `python -m pip install -e .` and
`python -m pip install jupyterlab ipykernel` if needed.
The notebook works from the repository root or `scripts/`. Run All requires internet
access and downloads fresh history. Download errors stop execution rather than
substituting invented prices.

**Demo isolation:** all instrument/price Parquet files and portfolio exports are
created under a fresh system temporary directory. Existing repository data files
are never used. Temporary files are deleted when the kernel exits or setup reruns.

Tables use `print()` for ConvexPM terminal formatting. Dedicated interactive
charts are embedded below. In Jupyter, trust the notebook to enable chart HTML
and JavaScript; rerunning the chart cells regenerates their output.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import os
import pandas as pd
demo_workspace = TemporaryDirectory(prefix="convexpm-demo-")
DEMO_ROOT = Path(demo_workspace.name)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "convexpm").is_dir()), None)
if ROOT is not None and str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yfinance as yf
yf.set_tz_cache_location(str(DEMO_ROOT / "yahoo-cache"))

from lightweight_charts import JupyterChart

from convexpm import Instrument, InstrumentRegistry, MarketDataStore, Portfolio, Trade, instrument_plot
from convexpm.importers import IBKRTransactionParser, Renta4ImportParser
from convexpm.market_data import MarketDataUpdater, YahooProvider
from convexpm.transactions import trades_to_dataframe

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.4f}".format)
START, END = "2024-01-02", "2026-09-30"
print(f"Demo period: {START} to {END}; Yahoo prices, invented trades.")
print("Storage: isolated temporary demo directory")


Demo period: 2024-01-02 to 2026-09-30; Yahoo prices, invented trades.
Storage: isolated temporary demo directory


## Functions used in explore_my_portfolio_2.py

The source is an executable scratchpad rather than a module of function definitions.
This cell imports its public APIs and lists the calls to explore. Subsequent cells
show their outputs. Importing the scratchpad itself would run its personal workflow.
Yahoo prices are downloaded into temporary demo stores; broker imports are opt-in.


In [2]:
function_reference = pd.DataFrame([
    ("Instruments", "Instrument; InstrumentRegistry; add / update / remove / save; add_yahoo", "Define, persist, and look up securities"),
    ("Market data", "MarketDataStore; upsert / summary / delete_instrument; [id].history", "Store prices and inspect coverage"),
    ("Instrument statistics", "[id].stats; stats(periods, freq='M' or 'Q')", "Returns, volatility, drawdown, and period statistics"),
    ("Instrument charts", "instrument_plot; add_instrument / remove_instrument / relative_chart / absolute_chart / close", "Dedicated instrument comparison chart"),
    ("Downloads", "MarketDataUpdater; YahooProvider; update_instrument", "Incremental updates or forced backfills"),
    ("Broker imports", "Renta4ImportParser.parse_directory; IBKRTransactionParser.parse; result.instruments", "Parse trades, metadata, and available price histories"),
    ("Trades", "Trade; trades_to_dataframe; add_trades(cash_ledger=True); deduplicate_trades; trades.delete", "Record, inspect, and edit transactions"),
    ("Portfolio", "Portfolio; load / save / summary / chart / set_benchmark / performance_view", "Value positions in EUR and compare performance"),
    ("Performance tables", "summary; asset_weights / asset_net_inflows / asset_valuation", "Annual results, allocation, flows, and valuations"),
    ("Standalone metrics", "standalone_asset_returns / volatility / alpha / beta / drawdown", "Each asset while held, ignoring its portfolio weight"),
    ("Attribution", "return_contribution / volatility_contribution / drawdown_contribution", "Each asset's weighted contribution"),
    ("Correlations", "return_correlation / volatility_correlation; metadata", "Co-movement and calculation conventions"),
], columns=["Area", "Functions / accessors", "Purpose"])
print(function_reference)


                     Area                              Functions / accessors  \
0             Instruments  Instrument; InstrumentRegistry; add / update /...   
1             Market data  MarketDataStore; upsert / summary / delete_ins...   
2   Instrument statistics        [id].stats; stats(periods, freq='M' or 'Q')   
3       Instrument charts  instrument_plot; add_instrument / remove_instr...   
4               Downloads  MarketDataUpdater; YahooProvider; update_instr...   
5          Broker imports  Renta4ImportParser.parse_directory; IBKRTransa...   
6                  Trades  Trade; trades_to_dataframe; add_trades(cash_le...   
7               Portfolio  Portfolio; load / save / summary / chart / set...   
8      Performance tables  summary; asset_weights / asset_net_inflows / a...   
9      Standalone metrics  standalone_asset_returns / volatility / alpha ...   
10            Attribution  return_contribution / volatility_contribution ...   
11           Correlations  return_correl

## 1. Real stocks and Yahoo market history

The example investor holds Apple, Microsoft, Johnson & Johnson, Repsol, and
Banco Santander. SPY is the comparison benchmark, converted into EUR for
performance comparisons. USD/EUR means **EUR per USD**.

Prices use Yahoo's unadjusted daily **Close**, matching the source script's default.
The demo does not book dividend cash receipts; its performance is therefore a
price-based illustration, rather than a total-return statement. Trade amounts use
the most recent close on or before the invented trade date. Different exchanges
have different holidays; ConvexPM uses the latest available price for valuation.


In [3]:
my_instruments = InstrumentRegistry(DEMO_ROOT / "instruments.parquet", auto_load=False)
market_data = MarketDataStore(DEMO_ROOT / "market_data.parquet", registry=my_instruments)
instrument_specs = [
    ("AAPL", "Apple Inc.", "equity", "equity", "USD", "AAPL"),
    ("MSFT", "Microsoft Corporation", "equity", "equity", "USD", "MSFT"),
    ("JNJ", "Johnson & Johnson", "equity", "equity", "USD", "JNJ"),
    ("REP", "Repsol, S.A.", "equity", "equity", "EUR", "REP.MC"),
    ("SAN", "Banco Santander, S.A.", "equity", "equity", "EUR", "SAN.MC"),
    ("SPY", "SPDR S&P 500 ETF Trust", "etf", "equity", "USD", "SPY"),
    ("USDEUR", "USD/EUR FX Rate", "fx", "fx", "EUR", "USDEUR=X"),
]
for instrument_id, name, kind, asset_class, currency, symbol in instrument_specs:
    metadata = {"yahoo_symbol": symbol}
    if kind == "fx":
        metadata.update(base_currency="USD", quote_currency="EUR", meaning="1 USD in EUR")
    my_instruments.add(Instrument(instrument_id=instrument_id, name=name,
        instrument_type=kind, asset_class=asset_class, currency=currency,
        data_source="yahoo", data_symbol=symbol, metadata=metadata))
my_instruments.save()
print(pd.DataFrame([i.to_dict() for i in my_instruments.all()]).drop(columns="metadata"))


  instrument_id                    name instrument_type asset_class currency  \
0          AAPL              Apple Inc.          equity      equity      USD   
1           JNJ       Johnson & Johnson          equity      equity      USD   
2          MSFT   Microsoft Corporation          equity      equity      USD   
3           REP            Repsol, S.A.          equity      equity      EUR   
4           SAN   Banco Santander, S.A.          equity      equity      EUR   
5           SPY  SPDR S&P 500 ETF Trust             etf      equity      USD   
6        USDEUR         USD/EUR FX Rate              fx          fx      EUR   

  data_source data_symbol  
0       yahoo        AAPL  
1       yahoo         JNJ  
2       yahoo        MSFT  
3       yahoo      REP.MC  
4       yahoo      SAN.MC  
5       yahoo         SPY  
6       yahoo    USDEUR=X  


In [4]:
# Every updater is explicitly attached to the temporary demo stores.
updater = MarketDataUpdater(my_instruments, market_data, {"yahoo": YahooProvider()})
# Yahoo treats end as exclusive: add one day to include END.
fetch_end = (pd.Timestamp(END) + pd.Timedelta(days=1)).date()
download_summary = []
for instrument in my_instruments.all():
    fetched = updater.update_instrument(instrument.instrument_id,
        start=pd.Timestamp(START).date(), end=fetch_end, force=True)
    if fetched.empty or fetched["date"].min() > pd.Timestamp(START):
        raise RuntimeError(f"Missing initial history for {instrument.data_symbol}")
    if fetched["date"].max() < pd.Timestamp(END):
        raise RuntimeError(f"Incomplete history for {instrument.data_symbol}")
    download_summary.append({"symbol": instrument.data_symbol, "rows": len(fetched),
                             "first_date": fetched["date"].min(), "last_date": fetched["date"].max()})
print("Downloaded Yahoo daily Close prices into temporary demo stores.")
print(pd.DataFrame(download_summary))
print(market_data.summary())


Downloaded Yahoo daily Close prices into temporary demo stores.
     symbol  rows first_date  last_date
0      AAPL   689 2024-01-02 2026-09-30
1       JNJ   689 2024-01-02 2026-09-30
2      MSFT   689 2024-01-02 2026-09-30
3    REP.MC   702 2024-01-02 2026-09-30
4    SAN.MC   702 2024-01-02 2026-09-30
5       SPY   689 2024-01-02 2026-09-30
6  USDEUR=X   712 2024-01-02 2026-09-30
  instrument_id                    name    ticker asset_class instrument_type  \
0          AAPL              Apple Inc.      AAPL      equity          equity   
1           JNJ       Johnson & Johnson       JNJ      equity          equity   
2          MSFT   Microsoft Corporation      MSFT      equity          equity   
3           REP            Repsol, S.A.    REP.MC      equity          equity   
4           SAN   Banco Santander, S.A.    SAN.MC      equity          equity   
5           SPY  SPDR S&P 500 ETF Trust       SPY      equity             etf   
6        USDEUR         USD/EUR FX Rate  USDEUR=X

In [5]:
print(market_data["AAPL"].history.tail())
print(market_data["AAPL"].stats.summary)
print(market_data["AAPL"].stats(6, freq="M"))
print(market_data["JNJ"].stats(freq="Q"))


          date instrument_id    value currency value_type source
684 2026-09-24          AAPL 335.9200      USD      close  yahoo
685 2026-09-25          AAPL 341.0700      USD      close  yahoo
686 2026-09-28          AAPL 338.4000      USD      close  yahoo
687 2026-09-29          AAPL 329.4000      USD      close  yahoo
688 2026-09-30          AAPL 333.0200      USD      close  yahoo
observations                    689
first_date      2024-01-02 00:00:00
last_date       2026-09-30 00:00:00
first_price                185.6400
last_price                 333.0200
min_price                  165.0000
max_price                  341.0700
total_return                 0.7939
mean_return                  0.0010
volatility                   0.0174
sharpe                       0.9138
dtype: object
                          2026-03              2026-04              2026-05  \
observations                    1                   21                   20   
first_date    2026-03-31 00:00:00  2026-04

## 2. Invented trades and cash ledger

Initial capital is EUR 15,000 plus USD 12,000. EUR 500 monthly contributions and a
EUR 3,000 top-up fund later purchases. A EUR 1,500 withdrawal and partial sales
illustrate outflows and rebalancing. Each purchase uses a downloaded closing
price; fees are charged in the trade currency. `cash_ledger=True` generates the
opposite cash leg, so purchases use existing cash rather than new contributions.
Cash BUY/SELL rows represent external deposits/withdrawals in this demo.


In [7]:
def make_demo_trade(trade_id, dt, instrument_id, side, quantity, *, fees=0.0, note=""):
    instrument = my_instruments.get(instrument_id)
    price = 1.0 if instrument.instrument_type == "cash" else market_data.get_value(instrument_id, dt)
    return Trade(trade_id=trade_id, date=dt, instrument_id=instrument_id, side=side,
        quantity=quantity, amount=round(quantity * price, 2), value_per_unit=price,
        fees=fees, currency=instrument.currency, account="demo-broker",
        tags=["demo"], metadata={"source": "manual", "note": note})

# Explicit cash instruments make funding rows readable before insertion.
for currency in ("EUR", "USD"):
    my_instruments.add(Instrument(instrument_id=f"{currency}_CASH", name=f"{currency} Cash",
        instrument_type="cash", asset_class="cash", currency=currency, data_source="demo"))
manual_trades = [
    make_demo_trade("fund-eur", START, "EUR_CASH", "BUY", 15000, note="Initial EUR savings"),
    make_demo_trade("fund-usd", START, "USD_CASH", "BUY", 12000, note="Initial USD savings"),
]
orders = [
    ("2024-01-03", "AAPL", "BUY", 15, 1),
    ("2024-01-03", "JNJ", "BUY", 10, 1),
    ("2024-01-04", "REP", "BUY", 150, 3),
    ("2024-01-04", "MSFT", "BUY", 8, 1),
    ("2024-01-05", "SAN", "BUY", 600, 3),
    ("2024-06-14", "AAPL", "BUY", 4, 1),
    ("2024-09-16", "JNJ", "BUY", 5, 1),
    ("2025-02-14", "MSFT", "BUY", 3, 1),
    ("2025-04-15", "AAPL", "BUY", 3, 1),
    ("2025-08-15", "REP", "SELL", 50, 3),
    ("2025-08-18", "SAN", "BUY", 200, 3),
    ("2026-02-16", "REP", "BUY", 80, 3),
    ("2026-06-15", "MSFT", "SELL", 3, 1),
    ("2026-06-16", "SAN", "BUY", 150, 3),
]
for number, (dt, instrument_id, side, quantity, fees) in enumerate(orders, 1):
    manual_trades.append(make_demo_trade(f"order-{number:02d}", dt, instrument_id, side,
        quantity, fees=fees, note="Rebalance" if side == "SELL" else "Long-term investment"))
for month in pd.date_range("2024-02-01", "2026-09-01", freq="MS"):
    dt = (month + pd.offsets.BDay(0)).date()
    manual_trades.append(make_demo_trade(f"monthly-{month:%Y-%m}", dt,
        "EUR_CASH", "BUY", 500, note="Monthly contribution"))
manual_trades.extend([
    make_demo_trade("top-up", "2025-04-01", "EUR_CASH", "BUY", 3000, note="Savings top-up"),
    make_demo_trade("withdrawal", "2026-07-01", "EUR_CASH", "SELL", 1500, note="Holiday withdrawal"),
])
my_portfolio = Portfolio(name="Yahoo stock demo portfolio", base_currency="EUR",
                         registry=my_instruments, market_data=market_data)
inserted_count = my_portfolio.add_trades(manual_trades, cash_ledger=True)
print(f"Invented source trades: {len(manual_trades)}; inserted rows including cash legs: {inserted_count}")
print(trades_to_dataframe(manual_trades).sort_values(["date", "trade_id"]).head(15))


Invented source trades: 50; inserted rows including cash legs: 64
           trade_id       date instrument_id side  quantity      amount  \
0          fund-eur 2024-01-02      EUR_CASH  BUY     15000 15,000.0000   
1          fund-usd 2024-01-02      USD_CASH  BUY     12000 12,000.0000   
2          order-01 2024-01-03          AAPL  BUY        15  2,763.7500   
3          order-02 2024-01-03           JNJ  BUY        10  1,609.7000   
4          order-03 2024-01-04           REP  BUY       150  2,057.2500   
5          order-04 2024-01-04          MSFT  BUY         8  2,943.5200   
6          order-05 2024-01-05           SAN  BUY       600  2,347.5000   
16  monthly-2024-02 2024-02-01      EUR_CASH  BUY       500    500.0000   
17  monthly-2024-03 2024-03-01      EUR_CASH  BUY       500    500.0000   
18  monthly-2024-04 2024-04-01      EUR_CASH  BUY       500    500.0000   
19  monthly-2024-05 2024-05-01      EUR_CASH  BUY       500    500.0000   
20  monthly-2024-06 2024-06-03    

In [8]:
# Inspect both the source trades and generated cash legs.
print(my_portfolio.trades.to_dataframe().head(14))
print("Rows inserted by repeating the same import:", my_portfolio.add_trades(manual_trades, cash_ledger=True))
print("Duplicates removed:", my_portfolio.deduplicate_trades())


                        trade_id       date instrument_id  \
trade_id                                                    
fund-eur                fund-eur 2024-01-02      EUR_CASH   
fund-usd                fund-usd 2024-01-02      USD_CASH   
order-01                order-01 2024-01-03          AAPL   
order-01:cash      order-01:cash 2024-01-03      USD_CASH   
order-02                order-02 2024-01-03           JNJ   
order-02:cash      order-02:cash 2024-01-03      USD_CASH   
order-03                order-03 2024-01-04           REP   
order-03:cash      order-03:cash 2024-01-04      EUR_CASH   
order-04                order-04 2024-01-04          MSFT   
order-04:cash      order-04:cash 2024-01-04      USD_CASH   
order-05                order-05 2024-01-05           SAN   
order-05:cash      order-05:cash 2024-01-05      EUR_CASH   
monthly-2024-02  monthly-2024-02 2024-02-01      EUR_CASH   
monthly-2024-03  monthly-2024-03 2024-03-01      EUR_CASH   

                       

## 3. Portfolio valuation in EUR

USD holdings retain their native currency in the summary. `local_market_value`
is converted using `fx_rate` into EUR `market_value`. Weights include uninvested cash.


In [9]:
print(my_portfolio)

Yahoo stock demo portfolio BASE EUR
NAV: 56,748.70 EUR
instrument_id  name                   asset_class  quantity   price   market_value  weight  currency  status
-------------  ---------------------  -----------  ---------  ------  ------------  ------  --------  ------
EUR_CASH       EUR Cash               cash         23,990.85    1.00     23,990.85   42.28  EUR       ok    
SAN            Banco Santander, S.A.  equity          950.00   12.33     11,711.60   20.64  EUR       ok    
AAPL           Apple Inc.             equity           22.00  293.65      6,460.23   11.38  EUR       ok    
REP            Repsol, S.A.           equity          180.00   29.93      5,387.40    9.49  EUR       ok    
MSFT           Microsoft Corporation  equity            8.00  452.26      3,618.08    6.38  EUR       ok    
JNJ            Johnson & Johnson      equity           15.00  233.44      3,501.60    6.17  EUR       ok    
USD_CASH       USD Cash               cash          2,357.69    0.88     

## 4. Benchmark performance and capital flows

The view uses historical holdings. Investment returns adjust for capital flows;
adding savings increases NAV without being investment profit. Standalone metrics
ignore allocation weights; contribution tables apply portfolio weights. Figures
are demo calculations, with conventions recorded in `performance.metadata`.


In [10]:
my_portfolio.set_benchmark("SPY")
performance = my_portfolio.performance_view(start=START, end=END)
print(performance.summary)


PERFORMANCE SUMMARY
index                             2024       2025       2026       Total    
--------------------------------  ---------  ---------  ---------  ---------
portfolio_return (%)                   6.33      12.40      13.52      35.67
benchmark_return (%)                  31.54       3.07      15.84      57.05
cagr (%)                               6.35      12.45      18.57      11.76

sharpe                                 1.13       1.59       2.42       1.68
benchmark_sharpe                       2.08       0.25       1.45       1.07
volatility (%)                         5.43       7.34       6.91       6.58
benchmark_volatility (%)              13.67      20.88      14.38      16.81
max_drawdown (%)                      -4.88      -8.39      -2.40      -8.39
benchmark_max_drawdown (%)            -8.64     -23.26      -8.08     -23.26

alpha (%)                             -3.04      11.44       8.47      18.49
beta                                   0.29       0.26

### `asset_weights`

Allocation at each year end and the end of the full period (%).


In [11]:
print(performance.asset_weights)


ASSET WEIGHTS
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                13.73     10.83     11.38      11.38
EUR Cash                  48.32     51.29     42.28      42.28
Johnson & Johnson          6.26      5.62      6.17       6.17
Microsoft Corporation      9.73      9.63      6.38       6.38
Repsol, S.A.               5.27      3.39      9.49       9.49
Banco Santander, S.A.      8.05     17.14     20.64      20.64
USD Cash                   8.64      2.10      3.66       3.66
=====================  ========  ========  ========  =========
Portfolio Weight         100.00    100.00    100.00     100.00


### `asset_net_inflows`

Net investment by asset in EUR; the total reconciles portfolio net capital flows.


In [12]:
print(performance.asset_net_inflows)


ASSET NET INFLOWS
asset                  2024       2025       2026       Total    
---------------------  ---------  ---------  ---------  ---------
Apple Inc.              3,319.20     535.66       0.00   3,854.86
EUR Cash               16,089.25   8,021.90    -120.30  23,990.85
Johnson & Johnson       2,225.96       0.00       0.00   2,225.96
Microsoft Corporation   2,694.53   1,172.17  -1,032.73   2,833.98
Repsol, S.A.            2,060.25    -665.50   1,372.20   2,766.95
Banco Santander, S.A.   2,350.50   1,643.60   1,748.10   5,742.20
USD Cash                2,631.11  -1,707.83   1,032.73   1,956.00
=====================  =========  =========  =========  =========
Portfolio Net Inflow   31,370.80   9,000.00   3,000.00  43,370.80


### `asset_valuation`

Year-end asset values in EUR, including cash.


In [13]:
print(performance.asset_valuation)


ASSET VALUATION
asset                  2024       2025       2026       Total    
---------------------  ---------  ---------  ---------  ---------
Apple Inc.              4,572.28   5,091.32   6,460.23   6,460.23
EUR Cash               16,089.25  24,111.15  23,990.85  23,990.85
Johnson & Johnson       2,084.63   2,642.52   3,501.60   3,501.60
Microsoft Corporation   3,240.39   4,528.55   3,618.08   3,618.08
Repsol, S.A.            1,753.50   1,592.50   5,387.40   5,387.40
Banco Santander, S.A.   2,678.70   8,056.00  11,711.60  11,711.60
USD Cash                2,876.30     986.96   2,078.94   2,078.94
=====================  =========  =========  =========  =========
Portfolio Valuation    33,295.05  47,009.00  56,748.70  56,748.70


### `standalone_asset_returns`

Returns for each asset while held, in portfolio base currency (%).


In [14]:
print(performance.standalone_asset_returns)


STANDALONE ASSET RETURNS
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                42.91     -3.83     26.89      74.38
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson         -5.53     26.76     32.51      58.68
Microsoft Corporation     20.30      1.64      9.86      34.32
Repsol, S.A.             -14.76     36.23     87.94     118.23
Banco Santander, S.A.     14.11    125.56     22.42     215.09
USD Cash                  -0.45     -2.22     -2.27      -4.86


### `standalone_asset_volatility`

Annualized volatility of each held asset (%).


In [15]:
print(performance.standalone_asset_volatility)


STANDALONE ASSET VOLATILITY
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                23.46     32.27     27.26      27.95
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson         16.49     19.54     21.24      19.01
Microsoft Corporation     20.50     24.98     35.27      26.79
Repsol, S.A.              18.95     24.25     35.92      26.49
Banco Santander, S.A.     25.52     31.27     31.55      29.43
USD Cash                   3.68      9.67      2.59       6.38


### `standalone_asset_alpha`

Standalone alpha relative to the chosen benchmark (%).


In [16]:
print(performance.standalone_asset_alpha)


STANDALONE ASSET ALPHA
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                10.01     -7.45     14.89      13.77
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson        -11.08     26.39     31.73      51.92
Microsoft Corporation    -16.68     -1.10     -3.95     -20.28
Repsol, S.A.             -24.37     32.35     92.22     102.71
Banco Santander, S.A.      5.62    124.02      3.52     167.32
USD Cash                  -0.66     -2.20     -1.93      -4.56


### `standalone_asset_beta`

Standalone sensitivity to benchmark returns.


In [17]:
print(performance.standalone_asset_beta)


STANDALONE ASSET BETA
asset                  2024  2025   2026   Total
---------------------  ----  -----  -----  -----
Apple Inc.             1.05   1.18   0.76   1.06
EUR Cash               0.00   0.00   0.00   0.00
Johnson & Johnson      0.18   0.12   0.05   0.12
Microsoft Corporation  1.17   0.89   0.87   0.95
Repsol, S.A.           0.25   0.16  -0.38   0.07
Banco Santander, S.A.  0.19   0.33   0.72   0.37
USD Cash               0.01  -0.00  -0.02  -0.01


### `standalone_asset_drawdown`

Standalone maximum drawdown (%).


In [18]:
print(performance.standalone_asset_drawdown)


STANDALONE ASSET DRAWDOWN
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.               -13.87    -34.36    -12.47     -36.58
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson        -11.41    -19.01    -11.79     -19.01
Microsoft Corporation    -16.41    -26.57    -25.23     -33.74
Repsol, S.A.             -31.90    -27.04    -20.45     -40.77
Banco Santander, S.A.    -18.74    -20.79    -17.19     -20.79
USD Cash                  -3.15     -7.74     -2.27      -9.84


### `return_contribution`

Weighted contributions to portfolio return (%).


In [19]:
print(performance.return_contribution)


RETURN CONTRIBUTION
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                 4.02     -0.82      2.75       6.94
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson         -0.46      1.29      1.86       2.75
Microsoft Corporation      2.07      0.46     -0.16       3.01
Repsol, S.A.              -0.90      1.28      5.04       5.34
Banco Santander, S.A.      1.27     10.03      4.08      17.08
USD Cash                   0.33      0.16     -0.05       0.55
=====================  ========  ========  ========  =========
Portfolio Return           6.33     12.40     13.52      35.67


### `volatility_contribution`

Weighted contributions to portfolio volatility.


In [20]:
print(performance.volatility_contribution)


VOLATILITY CONTRIBUTION
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                35.05     34.15     25.50      31.71
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson          4.72      2.61      4.00       3.58
Microsoft Corporation     27.24     24.14     14.93      22.07
Repsol, S.A.              11.04      6.29     -1.76       5.14
Banco Santander, S.A.     20.41     32.28     57.37      36.90
USD Cash                   1.54      0.54     -0.04       0.60
=====================  ========  ========  ========  =========
Portfolio Volatility       5.43      7.34      6.91       6.58


### `drawdown_contribution`

Asset contributions to portfolio drawdown.


In [21]:
print(performance.drawdown_contribution)


DRAWDOWN CONTRIBUTION
asset                  2024 (%)  2025 (%)  2026 (%)  Total (%)
---------------------  --------  --------  --------  ---------
Apple Inc.                -1.61     -3.53     -0.76      -3.53
EUR Cash                   0.00      0.00      0.00       0.00
Johnson & Johnson          0.21     -0.67      0.01      -0.67
Microsoft Corporation     -1.69     -2.10     -0.69      -2.10
Repsol, S.A.              -0.77     -1.20      1.90      -1.20
Banco Santander, S.A.     -1.04     -0.88     -2.86      -0.88
USD Cash                   0.00      0.00      0.00       0.00
Portfolio Drawdown        -4.88     -8.39     -2.40      -8.39


## 5. Correlations and calculation metadata

Filter correlations to a year or a start/end range. Cash has constant EUR returns,
so some correlation entries may be undefined rather than indicating co-movement.

Each correlation table prints the first and last observation dates actually used
after filtering. A partial year shows only its available dates.


In [28]:
print(performance.volatility_correlation())
print()

print(performance.volatility_correlation(year=2026))
print()

print(performance.return_correlation())
print()

print(performance.return_correlation(start="2025-01-01", end="2025-12-31"))


VOLATILITY CORRELATION
3/Jan/2024 - 30/Sep/2026
index     AAPL   EUR_CASH  JNJ   MSFT   REP.MC  SAN.MC  USD_CASH
--------  -----  --------  ----  -----  ------  ------  --------
AAPL       1.00      0.00  0.14   0.31    0.13    0.18     -0.02
EUR_CASH   0.00      0.00  0.00   0.00    0.00    0.00      0.00
JNJ        0.14      0.00  1.00   0.18    0.06    0.03      0.01
MSFT       0.31      0.00  0.18   1.00    0.08    0.09     -0.02
REP.MC     0.13      0.00  0.06   0.08    1.00    0.25      0.01
SAN.MC     0.18      0.00  0.03   0.09    0.25    1.00      0.05
USD_CASH  -0.02      0.00  0.01  -0.02    0.01    0.05      1.00

VOLATILITY CORRELATION
1/Jan/2026 - 30/Sep/2026
index     AAPL   EUR_CASH  JNJ    MSFT   REP.MC  SAN.MC  USD_CASH
--------  -----  --------  -----  -----  ------  ------  --------
AAPL       1.00      0.00   0.23   0.12   -0.07   -0.04      0.02
EUR_CASH   0.00      0.00   0.00   0.00    0.00    0.00      0.00
JNJ        0.23      0.00   1.00   0.24   -0.04   -0.1

## 6. Dedicated ConvexPM charts

These are the same chart APIs used in `explore_my_portfolio_2.py`, rendered inline
with `JupyterChart`. Hover, zoom, and pan to explore the price series and risk bands.
The instrument chart compares stocks in EUR. The portfolio NAV chart includes
cumulative net contributions, volatility envelopes, and a drawdown tolerance floor.

Charts require a trusted notebook viewer with HTML/JavaScript support. They use
only the temporary demo stores. Tables use `print()` to retain terminal styling.


In [23]:
# Dedicated instrument comparison chart, embedded in Jupyter.
instrument_widget = JupyterChart(width=1000, height=450)
price_chart = instrument_plot(start=START, end=END, curr="EUR",
    registry=my_instruments, market_data=market_data,
    chart=instrument_widget, show=False)
for instrument_id in ("AAPL", "MSFT", "JNJ", "REP", "SAN"):
    price_chart.add_instrument(instrument_id)
price_chart.relative_chart()
# To display absolute EUR prices instead, call price_chart.absolute_chart()
# before instrument_widget.load(). Rerun this cell to regenerate the widget.
instrument_widget.load()


In [24]:
# Dedicated portfolio NAV chart, including the native contribution and risk bands.
nav_widget = JupyterChart(width=1000, height=450)
portfolio_chart = my_portfolio.chart(start=START, end=END, horizon=90,
    drawdown_tolerance=0.15, chart=nav_widget, show=False)
nav_widget.load()


## 7. Save, reload, and delete a trade

Persistence uses the isolated demo directory. A disposable cash trade demonstrates
selection and deletion without changing the performance results above. Deleting a
funded security order also requires deleting its generated `:cash` trade.


In [25]:
my_instruments.save()
my_portfolio.save("demo", root=DEMO_ROOT / "portfolios")
reloaded = Portfolio.load("demo", root=DEMO_ROOT / "portfolios",
                         registry=my_instruments, market_data=market_data)
print(f"Reloaded {reloaded.name}: {len(reloaded.trades)} rows; benchmark={reloaded.benchmark}")
print(f"Reloaded NAV: {reloaded.nav(END):,.2f} EUR")
reloaded.add_trades([make_demo_trade("disposable", END, "EUR_CASH", "BUY", 100,
                                   note="Temporary example")])
print(reloaded.trades["instrument_id"] == "EUR_CASH")
deleted_trade = reloaded.trades.delete("disposable")
print(f"Deleted: {deleted_trade.trade_id}; NAV restored: {reloaded.nav(END):,.2f} EUR")


Reloaded Yahoo stock demo portfolio: 64 rows; benchmark=SPY
Reloaded NAV: 56,748.70 EUR
trade_id
fund-eur            True
fund-usd           False
order-01           False
order-01:cash      False
order-02           False
                   ...  
monthly-2026-07     True
withdrawal          True
monthly-2026-08     True
monthly-2026-09     True
disposable          True
Name: instrument_id, Length: 65, dtype: bool
Deleted: disposable; NAV restored: 56,748.70 EUR


## 8. Broker imports (opt-in) and incremental Yahoo updates

The broker cell mirrors the original script. Supply your own paths and enable its
flag to inspect imports; these records are not merged into the demo portfolio.

### Incremental Yahoo updates

The main download cell above runs automatically. This cell demonstrates a second
update over the same range: it should insert zero rows because the requested
history is already stored. All calls use the same isolated temporary demo stores.


In [26]:
RUN_BROKER_IMPORTS = False
if RUN_BROKER_IMPORTS:
    R4_DIR = Path("/path/to/renta4/files")
    IBKR_CSV = Path("/path/to/ibkr/transactions.csv")
    r4_import = Renta4ImportParser().parse_directory(R4_DIR)
    print(trades_to_dataframe(r4_import.trades))
    print(r4_import.market_data.head())
    print("Renta 4 cash rows:", len(r4_import.cash_transactions))
    print("Morningstar reports:", len(r4_import.morningstar_reports))
    ibkr_import = IBKRTransactionParser().parse(IBKR_CSV)
    ibkr_instruments = ibkr_import.instruments(yahoo_symbols={}, known_ids=set(),
                                              fetch_yahoo_metadata=False)
    print(pd.DataFrame([i.to_dict() for i in ibkr_instruments]))
    print(trades_to_dataframe([*ibkr_import.trades, *ibkr_import.cash_trades]))
else:
    print("Broker examples disabled; no personal files needed for the demo.")


Broker examples disabled; no personal files needed for the demo.


In [27]:
additional_rows = updater.update_instrument("AAPL", end=pd.Timestamp(END).date())
print(f"Rows fetched by a repeated incremental update: {len(additional_rows)}")
# Optional cleanup operations, scoped only to these temporary demo stores:
# market_data.delete_instrument("AAPL")
# my_instruments.remove("AAPL", missing_ok=True)
# my_instruments.save()


Rows fetched by a repeated incremental update: 0
